# 6.14 Confusion Matrix

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.14_confusion-matrix.ipynb)

Companion code for Section 6.14 of the book. Run the cells in order from top to bottom.

**Example 6.14.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, matthews_corrcoef,
                              cohen_kappa_score, balanced_accuracy_score)
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Binary confusion matrix ───────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
y_pred = model.predict(X_te)

cm = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=== Binary Confusion Matrix ===')
print(f'True Negatives  (TN): {tn:4d}  - Correctly predicted malignant')
print(f'False Positives (FP): {fp:4d}  - Malignant predicted as benign (DANGEROUS!)')
print(f'False Negatives (FN): {fn:4d}  - Benign predicted as malignant (annoying)')
print(f'True Positives  (TP): {tp:4d}  - Correctly predicted benign')
print()
print(f'Accuracy:             {(tp+tn)/(tp+tn+fp+fn):.4f}')
print(f'Precision:            {tp/(tp+fp):.4f}  (PPV)')
print(f'Recall/Sensitivity:   {tp/(tp+fn):.4f}  (TPR)')
print(f'Specificity:          {tn/(tn+fp):.4f}  (TNR)')
print(f'False Positive Rate:  {fp/(fp+tn):.4f}  (FPR = 1-Specificity)')
print(f'MCC:                  {matthews_corrcoef(y_te, y_pred):.4f}  (Matthews Correlation)')
print(f'Cohen Kappa:          {cohen_kappa_score(y_te, y_pred):.4f}')
print(f'Balanced Accuracy:    {balanced_accuracy_score(y_te, y_pred):.4f}')

# ── Multi-class confusion matrix ──────────────────────────────────
iris = load_iris()
X_i, y_i = iris.data, iris.target
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_i, y_i, test_size=0.2, stratify=y_i, random_state=42)
model_iris = RandomForestClassifier(n_estimators=100, random_state=42)
model_iris.fit(X_tr_i, y_tr_i)
y_pred_iris = model_iris.predict(X_te_i)

# ── Visualise both confusion matrices ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

# Binary
ax = axes[0]
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
           xticklabels=data.target_names, yticklabels=data.target_names,
           linewidths=2, cbar=False)
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
ax.set_title(f'Binary: Breast Cancer\nAccuracy={model.score(X_te,y_te):.3f}')

# Add percentage annotations
for i in range(2):
    for j in range(2):
        ax.text(j+0.5, i+0.75, f'({cm_norm[i,j]:.1%})', ha='center',
               va='center', fontsize=9, color='gray')

# Multi-class
ax = axes[1]
cm_iris = confusion_matrix(y_te_i, y_pred_iris)
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='Greens', ax=ax,
           xticklabels=iris.target_names, yticklabels=iris.target_names,
           linewidths=2, cbar=False)
ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
ax.set_title(f'Multi-class: Iris (3 classes)\nAccuracy={model_iris.score(X_te_i,y_te_i):.3f}')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()